In [13]:
import geopandas as gpd

In [14]:
gdf = gpd.read_file("land_use/hoodgdf.gpkg")
gdf

,Name,sidewalk_index,bike_index,cartway_density_index,ndvi_score,park_proximity_score,tree_score,health_proximity_score,service_proximity_score,geometry
0,LAWNDALE,0.981667,0.201329,0.633628,0.356519,0.886946,0.000000,0.871489,0.892003,"POLYGON ((-75.08616 40.05013, -75.08893 40.044..."
1,ASTON_WOODBRIDGE,0.946746,0.201768,0.682118,0.560973,0.437272,0.000000,0.788306,0.646258,"POLYGON ((-75.00860 40.05369, -75.00861 40.053..."
2,CARROLL_PARK,0.964426,0.190563,0.735923,0.266218,0.823277,0.064336,0.892550,0.879310,"POLYGON ((-75.22673 39.97720, -75.22022 39.974..."
3,CHESTNUT_HILL,0.950545,0.022636,0.439744,0.804823,0.713442,0.000000,0.810931,0.812973,"POLYGON ((-75.21278 40.08637, -75.21272 40.086..."
4,BURNHOLME,0.923102,0.145879,0.864792,0.566354,0.759824,0.044473,0.846795,0.806047,"POLYGON ((-75.08768 40.06861, -75.08758 40.068..."
...,...,...,...,...,...,...,...,...,...,...
153,DICKINSON_NARROWS,0.918205,0.096523,0.814234,0.104201,0.906231,0.000000,0.959764,0.738886,"POLYGON ((-75.15244 39.92716, -75.15294 39.924..."
154,GARDEN_COURT,1.012222,0.365182,0.926294,0.329481,0.950068,0.000000,0.802265,0.906192,"POLYGON ((-75.21415 39.95312, -75.21469 39.950..."
155,WISSAHICKON_HILLS,0.997497,0.436108,0.889194,0.587372,0.722529,0.000000,0.896466,0.819895,"POLYGON ((-75.22010 40.03757, -75.22064 40.037..."
156,DEARNLEY_PARK,0.814339,0.206014,0.811131,0.735065,0.692587,0.139064,0.739907,0.535791,"POLYGON ((-75.25136 40.04355, -75.25010 40.044..."


In [15]:
gdf_mobility= gpd.read_file("census/gdf_mobility.gpkg")

In [16]:
gdf = gdf.to_crs(epsg=4326)
gdf_mobility = gdf_mobility.to_crs(epsg=4326)

# This adds neighborhood info to each tract
gdf_mobility_with_neighborhood = gpd.sjoin(
    gdf_mobility,
    gdf[["Name", "geometry"]],  # neighborhood polygons
    how="left",
    predicate="intersects"          # tract centroid inside neighborhood polygon
)


In [17]:
# Average social variables per neighborhood
social_cols = ["disability_rate_inv", "age_65_plus_rate_inv", "commute_walk_rate"]

gdf_social = gdf_mobility_with_neighborhood.groupby("Name")[social_cols].mean().reset_index()


In [18]:
# gdf = neighborhood polygons
gdf = gdf.merge(gdf_social, on="Name", how="left")
gdf

,Name,sidewalk_index,bike_index,cartway_density_index,ndvi_score,park_proximity_score,tree_score,health_proximity_score,service_proximity_score,geometry,disability_rate_inv,age_65_plus_rate_inv,commute_walk_rate
0,LAWNDALE,0.981667,0.201329,0.633628,0.356519,0.886946,0.000000,0.871489,0.892003,"POLYGON ((-75.08616 40.05013, -75.08893 40.044...",0.276336,0.835755,0.000160
1,ASTON_WOODBRIDGE,0.946746,0.201768,0.682118,0.560973,0.437272,0.000000,0.788306,0.646258,"POLYGON ((-75.00860 40.05369, -75.00860 40.053...",0.237225,0.782365,0.025380
2,CARROLL_PARK,0.964426,0.190563,0.735923,0.266218,0.823277,0.064336,0.892550,0.879310,"POLYGON ((-75.22673 39.97720, -75.22022 39.974...",0.298271,0.818437,0.044255
3,CHESTNUT_HILL,0.950545,0.022636,0.439744,0.804823,0.713442,0.000000,0.810931,0.812973,"POLYGON ((-75.21278 40.08637, -75.21272 40.086...",0.264706,0.722637,0.032959
4,BURNHOLME,0.923102,0.145879,0.864792,0.566354,0.759824,0.044473,0.846795,0.806047,"POLYGON ((-75.08768 40.06861, -75.08758 40.068...",0.292346,0.795103,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,DICKINSON_NARROWS,0.918205,0.096523,0.814234,0.104201,0.906231,0.000000,0.959764,0.738886,"POLYGON ((-75.15243 39.92716, -75.15294 39.924...",0.234256,0.847950,0.399241
154,GARDEN_COURT,1.012222,0.365182,0.926294,0.329481,0.950068,0.000000,0.802265,0.906192,"POLYGON ((-75.21415 39.95312, -75.21469 39.950...",0.264848,0.865543,0.254623
155,WISSAHICKON_HILLS,0.997497,0.436108,0.889194,0.587372,0.722529,0.000000,0.896466,0.819895,"POLYGON ((-75.22010 40.03757, -75.22064 40.037...",0.275295,0.848807,0.009654
156,DEARNLEY_PARK,0.814339,0.206014,0.811131,0.735065,0.692587,0.139064,0.739907,0.535791,"POLYGON ((-75.25136 40.04356, -75.25010 40.044...",0.305790,0.780426,0.046604


In [19]:
# Mobility Score
gdf["mobility_score"] = gdf[[
    "cartway_density_index",  # curb_density_index
    "sidewalk_index",
    "bike_index"
]].mean(axis=1)

# Environmental Score
gdf["environmental_score"] = gdf[[
    "park_proximity_score",
    "tree_score",
    "ndvi_score"
]].mean(axis=1)

# Land Use Score
gdf["land_use_score"] = gdf[[
    "health_proximity_score",
    "service_proximity_score"
]].mean(axis=1)

# Social Score
gdf["social_score"] = gdf[[
    "age_65_plus_rate_inv",  # elderly index
    "disability_rate_inv",   # disability index
    "commute_walk_rate"      # walk commute index
]].mean(axis=1)


In [20]:
weights = {
    "mobility_score": 0.4,      # highest
    "land_use_score": 0.3,
    "environmental_score": 0.2,
    "social_score": 0.1          # lowest
}


In [21]:
gdf["accessibility_score"] = (
    gdf["mobility_score"] * weights["mobility_score"] +
    gdf["land_use_score"] * weights["land_use_score"] +
    gdf["environmental_score"] * weights["environmental_score"] +
    gdf["social_score"] * weights["social_score"]
)


In [22]:
import geopandas as gpd


# Columns example:
# ["NAME", "street_a", "street_b", "street_c",
#  "social_a", "social_b", "social_c",
#  "env_a", "env_b", "env_c",
#  "land_a", "land_b", "land_c"]
gdf

,Name,sidewalk_index,bike_index,cartway_density_index,ndvi_score,park_proximity_score,tree_score,health_proximity_score,service_proximity_score,geometry,disability_rate_inv,age_65_plus_rate_inv,commute_walk_rate,mobility_score,environmental_score,land_use_score,social_score,accessibility_score
0,LAWNDALE,0.981667,0.201329,0.633628,0.356519,0.886946,0.000000,0.871489,0.892003,"POLYGON ((-75.08616 40.05013, -75.08893 40.044...",0.276336,0.835755,0.000160,0.605541,0.414488,0.881746,0.370750,0.626713
1,ASTON_WOODBRIDGE,0.946746,0.201768,0.682118,0.560973,0.437272,0.000000,0.788306,0.646258,"POLYGON ((-75.00860 40.05369, -75.00860 40.053...",0.237225,0.782365,0.025380,0.610211,0.332748,0.717282,0.348324,0.560651
2,CARROLL_PARK,0.964426,0.190563,0.735923,0.266218,0.823277,0.064336,0.892550,0.879310,"POLYGON ((-75.22673 39.97720, -75.22022 39.974...",0.298271,0.818437,0.044255,0.630304,0.384610,0.885930,0.386988,0.633521
3,CHESTNUT_HILL,0.950545,0.022636,0.439744,0.804823,0.713442,0.000000,0.810931,0.812973,"POLYGON ((-75.21278 40.08637, -75.21272 40.086...",0.264706,0.722637,0.032959,0.470975,0.506088,0.811952,0.340101,0.567203
4,BURNHOLME,0.923102,0.145879,0.864792,0.566354,0.759824,0.044473,0.846795,0.806047,"POLYGON ((-75.08768 40.06861, -75.08758 40.068...",0.292346,0.795103,0.000000,0.644591,0.456883,0.826421,0.362483,0.633388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153,DICKINSON_NARROWS,0.918205,0.096523,0.814234,0.104201,0.906231,0.000000,0.959764,0.738886,"POLYGON ((-75.15243 39.92716, -75.15294 39.924...",0.234256,0.847950,0.399241,0.609654,0.336811,0.849325,0.493816,0.615403
154,GARDEN_COURT,1.012222,0.365182,0.926294,0.329481,0.950068,0.000000,0.802265,0.906192,"POLYGON ((-75.21415 39.95312, -75.21469 39.950...",0.264848,0.865543,0.254623,0.767900,0.426516,0.854229,0.461671,0.694899
155,WISSAHICKON_HILLS,0.997497,0.436108,0.889194,0.587372,0.722529,0.000000,0.896466,0.819895,"POLYGON ((-75.22010 40.03757, -75.22064 40.037...",0.275295,0.848807,0.009654,0.774266,0.436634,0.858180,0.377919,0.692279
156,DEARNLEY_PARK,0.814339,0.206014,0.811131,0.735065,0.692587,0.139064,0.739907,0.535791,"POLYGON ((-75.25136 40.04356, -75.25010 40.044...",0.305790,0.780426,0.046604,0.610495,0.522239,0.637849,0.377607,0.577761


In [23]:
import folium
from folium.features import GeoJsonTooltip


# Define variables by category
variables = {
    "Mobility": ["cartway_density_index", "sidewalk_index", "bike_index"],
    "Land Use": ["health_proximity_score", "service_proximity_score"],
    "Environmental": ["park_proximity_score", "tree_score", "ndvi_score"],
    "Social": ["age_65_plus_rate_inv", "disability_rate_inv", "commute_walk_rate"]
}

# ------------------------
# Create base map
# ------------------------
m = folium.Map(location=[39.95, -75.16], zoom_start=12, tiles="cartodbpositron")

# ------------------------
# FeatureGroups
# ------------------------
main_group = folium.FeatureGroup(name="Accessibility Score", show=True)
category_groups = {}
variable_groups = {}

for cat in variables:
    category_groups[cat] = folium.FeatureGroup(name=f"{cat} Score", show=False)
    for var in variables[cat]:
        variable_groups[var] = folium.FeatureGroup(name=f"{cat}: {var}", show=False)

# ------------------------
# Helper function to add Choropleth to a FeatureGroup
# ------------------------
def add_choropleth(gdf, column, feature_group, legend_name, cmap="YlOrRd"):
    choropleth = folium.Choropleth(
        geo_data=gdf,
        data=gdf,
        columns=["Name", column],
        key_on="feature.properties.Name",
        fill_color=cmap,
        fill_opacity=0.8,
        line_opacity=0.5,
        legend_name=legend_name
    )
    # Add choropleth to the FeatureGroup
    choropleth.geojson.add_to(feature_group)

    # Add tooltip showing all metrics
    tooltip_fields = ["Name", "accessibility_score", "mobility_score",
                      "land_use_score", "environmental_score", "social_score"]
    all_vars = sum([variables[cat] for cat in variables], [])
    tooltip_fields += all_vars

    tooltip_aliases = ["Neighborhood", "Accessibility Score", "Mobility Score",
                       "Land Use Score", "Environmental Score", "Social Score"]
    tooltip_aliases += [v.replace("_", " ").title() for v in all_vars]

    folium.GeoJson(
        gdf,
        style_function=lambda feature: {"fillColor": "transparent", "color": "transparent", "weight": 0},
        tooltip=GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=True,
            sticky=True
        )
    ).add_to(feature_group)

# ------------------------
# Add main map: Accessibility Score
# ------------------------
add_choropleth(gdf, "accessibility_score", main_group, "Accessibility Score", cmap="YlGnBu")

# ------------------------
# Add category maps
# ------------------------
for cat, fg in category_groups.items():
    add_choropleth(gdf, f"{cat.lower().replace(' ', '_')}_score", fg, f"{cat} Score")

# ------------------------
# Add individual variable maps
# ------------------------
for var, fg in variable_groups.items():
    # Optional: different color maps per category
    cmap = "YlOrRd"
    if var in variables["Environmental"]:
        cmap = "Greens"
    elif var in variables["Social"]:
        cmap = "PuBu"
    add_choropleth(gdf, var, fg, var.replace("_", " ").title(), cmap=cmap)

# ------------------------
# Add all FeatureGroups to map
# ------------------------
main_group.add_to(m)
for fg in category_groups.values():
    fg.add_to(m)
for fg in variable_groups.values():
    fg.add_to(m)

# ------------------------
# Add layer control
# ------------------------
folium.LayerControl(collapsed=False).add_to(m)

# ------------------------
# Save map
# ------------------------
m.save("interactive_accessibility_map.html")
print("Map saved as interactive_accessibility_map.html")


Map saved as interactive_accessibility_map.html


In [24]:
import folium
from folium.features import GeoJsonTooltip
from folium.plugins import FeatureGroupSubGroup

# ------------------------
# Base map
# ------------------------
m = folium.Map(location=[39.95, -75.16], zoom_start=12, tiles="cartodbpositron")

# ------------------------
# FeatureGroups
# ------------------------
main_group = folium.FeatureGroup(name="Accessibility Score", show=True)
category_groups = {}
variable_groups = {}

for cat in variables:
    # Main category group
    category_groups[cat] = folium.FeatureGroup(name=f"{cat} Score", show=False)
    # Variables as subgroups
    for var in variables[cat]:
        variable_groups[var] = FeatureGroupSubGroup(category_groups[cat], name=f"{var.replace('_',' ').title()}", show=False)

# ------------------------
# Helper function to add Choropleth with 0.8 transparency
# ------------------------
def add_choropleth(gdf, column, feature_group, legend_name, cmap="YlOrRd", opacity=0.8):
    choropleth = folium.Choropleth(
        geo_data=gdf,
        data=gdf,
        columns=["Name", column],
        key_on="feature.properties.Name",
        fill_color=cmap,
        fill_opacity=opacity,   # transparency
        line_opacity=0.5,
        legend_name=legend_name
    )
    choropleth.geojson.add_to(feature_group)

    # Tooltip
    tooltip_fields = ["Name", "accessibility_score", "mobility_score",
                      "land_use_score", "environmental_score", "social_score"]
    all_vars = sum([variables[cat] for cat in variables], [])
    tooltip_fields += all_vars

    tooltip_aliases = ["Neighborhood", "Accessibility Score", "Mobility Score",
                       "Land Use Score", "Environmental Score", "Social Score"]
    tooltip_aliases += [v.replace("_", " ").title() for v in all_vars]

    folium.GeoJson(
        gdf,
        style_function=lambda feature: {"fillColor": "transparent", "color": "transparent", "weight": 0},
        tooltip=GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=True,
            sticky=True
        )
    ).add_to(feature_group)

# ------------------------
# Main map
# ------------------------
add_choropleth(gdf, "accessibility_score", main_group, "Accessibility Score", cmap="YlGnBu", opacity=0.8)

# ------------------------
# Category maps
# ------------------------
for cat, fg in category_groups.items():
    add_choropleth(gdf, f"{cat.lower().replace(' ','_')}_score", fg, f"{cat} Score", opacity=0.8)

# ------------------------
# Variable maps (nested under categories)
# ------------------------
for cat in variables:
    for var in variables[cat]:
        fg = variable_groups[var]
        cmap = "YlOrRd"
        if var in variables["Environmental"]:
            cmap = "Greens"
        elif var in variables["Social"]:
            cmap = "PuBu"
        add_choropleth(gdf, var, fg, var.replace("_", " ").title(), cmap=cmap, opacity=0.8)

# ------------------------
# Add groups to map
# ------------------------
main_group.add_to(m)
for fg in category_groups.values():
    fg.add_to(m)
for fg in variable_groups.values():
    fg.add_to(m)

# ------------------------
# Layer control
# ------------------------
folium.LayerControl(collapsed=False).add_to(m)

# ------------------------
# Save map
# ------------------------
m.save("interactive_accessibility_map.html")
print("Map saved as interactive_accessibility_map.html")


Map saved as interactive_accessibility_map.html


In [25]:
import folium
from folium.features import GeoJsonTooltip

# ------------------------
# Base map
# ------------------------
m = folium.Map(location=[39.95, -75.16], zoom_start=12, tiles="cartodbpositron")

# ------------------------
# FeatureGroups
# ------------------------
main_group = folium.FeatureGroup(name="Accessibility Score", show=True)
category_groups = {}
for cat in variables:
    category_groups[cat] = folium.FeatureGroup(name=f"{cat} Score", show=False)

# ------------------------
# Helper function to add Choropleth with 0.8 transparency
# ------------------------
def add_choropleth(gdf, column, feature_group, legend_name, cmap="YlOrRd", opacity=0.8):
    choropleth = folium.Choropleth(
        geo_data=gdf,
        data=gdf,
        columns=["Name", column],
        key_on="feature.properties.Name",
        fill_color=cmap,
        fill_opacity=opacity,   # transparency
        line_opacity=0.5,
        legend_name=legend_name
    )
    choropleth.geojson.add_to(feature_group)

    # Tooltip showing all variables within the category
    tooltip_fields = ["Name", "accessibility_score", "mobility_score",
                      "land_use_score", "environmental_score", "social_score"]
    all_vars = sum([variables[cat] for cat in variables], [])
    tooltip_fields += all_vars

    tooltip_aliases = ["Neighborhood", "Accessibility Score", "Mobility Score",
                       "Land Use Score", "Environmental Score", "Social Score"]
    tooltip_aliases += [v.replace("_", " ").title() for v in all_vars]

    folium.GeoJson(
        gdf,
        style_function=lambda feature: {"fillColor": "transparent", "color": "transparent", "weight": 0},
        tooltip=GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=True,
            sticky=True
        )
    ).add_to(feature_group)

# ------------------------
# Add main map: Accessibility Score
# ------------------------
add_choropleth(gdf, "accessibility_score", main_group, "Accessibility Score", cmap="YlGnBu", opacity=0.8)

# ------------------------
# Add category maps
# ------------------------
for cat, fg in category_groups.items():
    cat_col = f"{cat.lower().replace(' ','_')}_score"
    add_choropleth(gdf, cat_col, fg, f"{cat} Score", opacity=0.8)

# ------------------------
# Add all groups to map
# ------------------------
main_group.add_to(m)
for fg in category_groups.values():
    fg.add_to(m)

# ------------------------
# Add layer control
# ------------------------
folium.LayerControl(collapsed=False).add_to(m)

# ------------------------
# Save map
# ------------------------
m.save("interactive_accessibility_map.html")
print("Map saved as interactive_accessibility_map.html")


Map saved as interactive_accessibility_map.html


In [33]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

# -------------------------------------
# 1. Define groups
# -------------------------------------
groups = {
    "Mobility": ["sidewalk_index", "bike_index", "cartway_density_index"],
    "Environmental": ["park_proximity_score", "tree_score", "ndvi_score"],
    "Land Use": ["health_proximity_score", "service_proximity_score"],
    "Social": ["age_65_plus_rate_inv", "disability_rate_inv", "commute_walk_rate"],
}

group_colors = {
    "Mobility": "#1f77b4",
    "Environmental": "#2ca02c",
    "Land Use": "#ff7f0e",
    "Social": "#9467bd",
}

# -------------------------------------
# 2. Order variables by group
# -------------------------------------
ordered_vars = []
group_labels = []

for g, vlist in groups.items():
    ordered_vars.extend(vlist)
    group_labels.extend([g] * len(vlist))

# -------------------------------------
# 3. Correlation matrix
# -------------------------------------
corr = gdf[ordered_vars].corr()

base_mask = np.ones_like(corr.values, dtype=bool)

# -------------------------------------
# 4. Create masks for group filtering
# -------------------------------------
group_masks = {}
for g, vlist in groups.items():
    selected = [var in vlist for var in ordered_vars]
    mask = np.outer(selected, selected)
    group_masks[g] = mask

# -------------------------------------
# 5. Build Heatmap with reversed RdBu colorscale
# -------------------------------------
heatmap = go.Heatmap(
    z=corr.values,
    x=ordered_vars,
    y=ordered_vars,
    colorscale="RdBu_r",   # reversed!
    zmin=-1,
    zmax=1,
    showscale=True,
    text=np.round(corr.values, 2),
    texttemplate="%{text}",
    hovertemplate="<b>%{x}</b><br><b>%{y}</b><br>Corr: %{z:.3f}<extra></extra>"
)

fig = go.Figure(data=[heatmap])

# -------------------------------------
# 6. Checkbox Buttons (moved right)
# -------------------------------------
buttons = [
    dict(
        label="Show All",
        method="update",
        args=[{"z": [np.where(base_mask, corr.values, np.nan)]}]
    )
]

for g, mask in group_masks.items():
    buttons.append(
        dict(
            label=g,
            method="update",
            args=[{"z": [np.where(mask, corr.values, np.nan)]}]
        )
    )

fig.update_layout(
    title="Interactive Group-Filtered Correlation Heatmap",
    width=900,
    height=900,
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            x=1.27,   # shifted to the far right
            y=1.00,   # slightly above
            showactive=True,
            buttons=buttons,
            bgcolor="white"
        )
    ]
)

# -------------------------------------
# 7. Add group color strip
# -------------------------------------
for i, gname in enumerate(group_labels):
    fig.add_shape(
        type="rect",
        x0=i - 0.5, x1=i + 0.5,
        y0=len(ordered_vars) - 0.5,
        y1=len(ordered_vars) - 0.3,
        fillcolor=group_colors[gname],
        line=dict(width=0),
        layer="above"
    )

fig.update_xaxes(tickangle=45)

fig.show()
